# QLoRA fine-tune: Qwen3-4B → crossword-generator SLM

Distills the (Claude + verifier + scaffolding) pipeline into one-shot generation. Trains on the enhanced non-hardcoded dataset (`data/sft_non_hardcoded_enhanced/`; chat: fixed system contract → size-specific `NxN` user prompt → verified assistant program, each carrying only its own size's template under a one-line technique header), **response-only loss**, dev for validation, `eval` held out for the base-vs-tuned test (see `colab_eval_non_hardcoded.ipynb`).

## 1. Install (pinned, Qwen3-capable snapshot)
Versions are **pinned**, not `>=`, on purpose: the current `trl` (1.x) **removed** `DataCollatorForCompletionOnlyLM` and **renamed** `SFTConfig(max_seq_length=)` -> `max_length=`, which would break cells 6-7 with `-U`. `trl==0.19.1` is the last release that supports **both** Qwen3 (needs `transformers>=4.51`) and the response-only collator this notebook relies on.

In [ ]:
!pip install -q -U 'transformers==4.53.*' 'trl==0.19.1' 'peft==0.16.*' 'bitsandbytes==0.46.*' 'accelerate==1.8.*' 'datasets==3.6.*'

> **Expected pip warning -- safe to ignore.** You'll likely see a resolver complaint that Colab's pre-installed `gradio` wants `huggingface-hub>=1.2` but `huggingface-hub 0.36.x` is installed. That 0.3x version is **required** by `transformers 4.53`, and `gradio` is **not used** anywhere in this notebook, so the conflict is cosmetic -- the install still succeeds. **Do not upgrade `huggingface-hub`** (it would break `transformers`).

## 1b. Preflight: confirm the GPU **before** training
Colab Pro only helps if you actually got a bigger card — Pro can still hand you a ~16 GB T4, which will OOM this config (especially the fp16 merge in cell 8). This prints the GPU name + free VRAM and warns if you're under ~20 GB. Want **L4 (24 GB)** or **A100 (40 GB)**: Runtime -> Change runtime type.

In [ ]:
import os
# Set BEFORE torch initializes CUDA: lets the allocator grow segments instead of
# fragmenting (the "reserved but unallocated" memory in OOM tracebacks).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import torch
assert torch.cuda.is_available(), "No GPU attached -- Runtime > Change runtime type > GPU (L4 or A100)."
free_b, total_b = torch.cuda.mem_get_info()
gb = 1024 ** 3
name = torch.cuda.get_device_properties(0).name
total_gb, free_gb = total_b / gb, free_b / gb
print(f"GPU: {name}")
print(f"VRAM: {total_gb:.1f} GB total | {free_gb:.1f} GB free")
print(f"bf16 supported: {torch.cuda.is_bf16_supported()}")

# 4-bit Qwen3-4B QLoRA (seq-len 8192) + the fp16 merge (~8 GB) wants >= ~20 GB;
# A100 40GB is comfortable, L4 24GB fits via the seq-len-reduced micro-batch (cell 3).
if total_gb < 20:
    print()
    print("=" * 64)
    print(f"WARNING: only {total_gb:.0f} GB VRAM -- this looks like a T4.")
    print("This config will likely OOM (especially the fp16 merge in cell 8).")
    print("Fix: Runtime > Change runtime type > L4 (24GB) or A100 (40GB),")
    print("     then Runtime > Restart session and rerun from cell 1.")
    print("=" * 64)
else:
    print(f"OK -- {total_gb:.0f} GB fits 4-bit Qwen3-4B QLoRA (can raise seq-len toward 8192).")

## 2. Get the data
Uses the **enhanced non-hardcoded** dataset `data/sft_non_hardcoded_enhanced/` (reliable positives `train/dev/eval.jsonl` + the `works_too_long.jsonl` subsection).
- **Clone your repo:** set `REPO_URL` below. **Commit `data/sft_non_hardcoded_enhanced/` first** — it's under `data/` so it's tracked (unlike gitignored `runs/`).
- **Upload:** put those `.jsonl` files in a Colab folder and set `DATA_DIR` to it.

Cell 5 assembles the training corpus from these files **in memory** — 7/9/11 only (15×15 excluded, too long). The `works_too_long` supplements at 9/11 are **filtered to the ≥2/3-reliable programs** (`wtl_keep.json`: each kept program produces a valid N×N grid in ≥2/3 of runs); 9×9 is topped up with those (all available), 11×11 gets all of them. The 7×7 `works_too_long` (never trained) supplies the held-out dev set — **files on disk are never modified**.

In [ ]:
REPO_URL = "https://github.com/Avaneesh-Ramesh-07/CrosswordSLM.git"   # <-- REQUIRED unless you set DATA_DIR to an upload
DATA_DIR = None            # <-- set to an uploaded folder to skip the clone
import os

if DATA_DIR is None:
    assert "<REPO>" not in REPO_URL, (
        "Set REPO_URL to your repo, OR set DATA_DIR to a folder with the enhanced dataset "
        "(train/dev/eval.jsonl + works_too_long.jsonl) uploaded via the Files panel."
    )
    if not os.path.exists("slm"):
        !git clone -q $REPO_URL slm
    # Non-hardcoded ENHANCED dataset (purified-palette validated, topic=vocabulary).
    # The training corpus is assembled from these files in cell 5 (files not modified).
    DATA_DIR = "slm/data/sft_non_hardcoded_enhanced"

for _f in ("train.jsonl", "works_too_long.jsonl", "wtl_keep.json"):
    assert os.path.exists(f"{DATA_DIR}/{_f}"), (
        f"missing {_f} in {DATA_DIR} -- commit data/sft_non_hardcoded_enhanced/ to the repo first")
print("data dir:", DATA_DIR, os.listdir(DATA_DIR))

## 3. Config
Hyperparameters. **The per-device batch is auto-tuned to the GPU** detected in cell 1b: the *effective* batch stays ~16 (the right convergence target for ~1.7k rows) while the *per-device* batch scales with VRAM **and** sequence length. Gradient checkpointing stays **on** (needed at these seq-lens). Wall-clock is set by rows×epochs×seq-len, not batch size — a bigger per-device batch just improves GPU utilization.

In [ ]:
# Qwen3-4B instruct. Confirm the exact HF id (alts: "Qwen/Qwen3-4B",
# "Qwen/Qwen3-4B-Instruct"). Start from Instruct for fast SFT.
model_name  = "Qwen/Qwen3-4B-Instruct-2507"  # base model to fine-tune FROM
adapter_dir = "qwen3-4b-crossword-qlora"      # OUTPUT: trained LoRA adapter dir (merged -> adapter_dir + "-merged")
output_dir  = "results"                       # trainer checkpoints + logs

# QLoRA / LoRA
lora_r, lora_alpha, lora_dropout = 32, 64, 0.05
# Qwen attention + MLP projections
target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

# Real max tokens (Qwen tokenizer) for the 7/9/11 corpus: ~7k, AFTER template specialization +
# header condensation -- each program now carries ONLY its own size template (not all of 7/9/11)
# under a one-line docstring, so rows are ~half their old size. 15x15 is EXCLUDED from training.
# 12288 leaves wide margin (0 truncation;
# verified by the token-length preflight below).
# NOTE: the max row is only ~7k tok, so you may lower this to 8192 with 0 truncation; it saves
# some activation memory but NOT the batch cap below (that is set by the logits-accuracy spike).
max_seq_length = 12288

num_train_epochs = 3   # learn the reliable generation algorithms (raise to 5-8 if under-fit)

# ---- throughput config: auto-tuned to the GPU detected in cell 1b ----
# Wall-clock is set by rows x epochs x seq-len, NOT batch size, so we hold the EFFECTIVE batch
# at ~16 via accumulation and keep the PER-DEVICE batch SMALL. Why small: trl computes a
# token-accuracy metric by materializing the FULL logits tensor [batch x seq x vocab(~152k)]
# plus an fp32 .contiguous() copy of it; group_by_length packs the longest rows into one batch,
# so at per-device batch 4 that copy alone is ~21 GB and OOMs even an 80 GB A100. Batch 2 halves
# the spike (fits with margin). Gradient checkpointing stays ON (activations otherwise OOM too).
# FASTER OPTION: pip install liger-kernel and set use_liger_kernel=True (cell 7) -- it fuses
# lm_head+cross-entropy so full logits are never materialized AND trl skips the accuracy copy,
# which lets you raise the per-device batch well above these values.
_vram = globals().get("total_gb", 16.0)   # from cell 1b; fallback = conservative 16GB
gradient_checkpointing = True
# Micro-batch capped by the logits-accuracy spike (batch x seq x 152k), NOT just activations.
# (SDPA attention -- set in cell 4 -- keeps attention memory O(seq); the spike is the LM head.)
if max_seq_length <= 8192:
    per_device_train_batch_size = 4 if _vram >= 76 else (2 if _vram >= 38 else 1)
elif max_seq_length <= 16384:
    per_device_train_batch_size = 2 if _vram >= 76 else 1     # 152k-vocab logits copy -> keep small
else:                                    # ~16k-24k seq-len (15x15 programs)
    per_device_train_batch_size = 1
gradient_accumulation_steps = max(1, round(16 / per_device_train_batch_size))  # hold effective ~16
per_device_eval_batch_size = per_device_train_batch_size
eff = per_device_train_batch_size * gradient_accumulation_steps
print(f"[auto] VRAM~{_vram:.0f}GB -> per_device_batch={per_device_train_batch_size}, "
      f"accum={gradient_accumulation_steps} (effective ~{eff}), "
      f"grad_checkpointing={gradient_checkpointing}")

learning_rate = 2e-4
lr_scheduler_type = "cosine"
warmup_ratio = 0.03
weight_decay = 0.0
logging_steps = 10

## 4. Load Qwen3-4B in 4-bit (QLoRA) + LoRA adapters

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training

# T4 (Colab free tier) has no bf16 -> fall back to fp16 automatically.
bf16_ok = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if bf16_ok else torch.float16
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (no GPU!)"
print(f"GPU: {gpu} | bf16 supported: {bf16_ok} -> compute dtype {compute_dtype}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name, quantization_config=bnb_config, device_map={"": 0},
    torch_dtype=compute_dtype, trust_remote_code=True,
    attn_implementation="sdpa",   # O(seq) attention memory -> long 11x11 seqs (~7k tok) fit
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=gradient_checkpointing,   # auto-set in cell 3
    # reentrant=True: Qwen3 saves a different tensor count on recompute, which trips
    # the non-reentrant checkpointer ("A different number of tensors was saved...").
    gradient_checkpointing_kwargs={"use_reentrant": True},
)

peft_config = LoraConfig(
    r=lora_r, lora_alpha=lora_alpha, lora_dropout=lora_dropout,
    target_modules=target_modules, bias="none", task_type="CAUSAL_LM",
)

## 5. Load data + render the Qwen chat template
Each row is `{messages:[system,user,assistant]}`. We render it with the model's own chat template into a `text` field; the response-only collator (next cell) then masks everything up to the assistant turn so loss is computed **only on the program**.

In [ ]:
import json, random, collections
from datasets import Dataset, DatasetDict

# Assemble the training corpus from the enhanced dataset WITHOUT modifying the files.
# 7/9/11 only (15x15 excluded -- its programs are ~27k tokens). Composition:
#   7x7   = all reliable positives
#   9x9   = positives + >=2/3-reliable works_too_long (all available; may not reach the 7x7 count)
#   11x11 = positives + ALL >=2/3-reliable works_too_long
# works_too_long = correct-but-flaky generators; the 9/11 supplements are FILTERED to programs
# that produced a valid NxN grid in >=2/3 of runs (wtl_keep.json), so we do not train on the
# ones that mostly empty out. The 7x7 works_too_long (never trained) supplies held-out dev.
SIZES = (7, 9, 11)
def _load(path):
    out = []
    for l in open(path, encoding="utf-8"):
        if not l.strip():
            continue
        r = json.loads(l); m = r["meta"]; sz = m["effective_spec"]["size"]
        if sz in SIZES:
            out.append({"messages": r["messages"], "size": sz, "hash": m.get("program_hash")})
    return out

pos = _load(f"{DATA_DIR}/train.jsonl") + _load(f"{DATA_DIR}/dev.jsonl") + _load(f"{DATA_DIR}/eval.jsonl")
wtl = _load(f"{DATA_DIR}/works_too_long.jsonl")
byw = {s: [r for r in wtl if r["size"] == s] for s in SIZES}
# keep only the >=2/3-reliable (program,size) works_too_long pairs for TRAINING (sizes 9/11)
_keep = {(h, s) for h, s in json.load(open(f"{DATA_DIR}/wtl_keep.json"))["keep"]}
w9raw  = [r for r in byw[9]  if (r["hash"], 9)  in _keep]
w11    = [r for r in byw[11] if (r["hash"], 11) in _keep]
n7  = sum(1 for r in pos if r["size"] == 7)
n9p = sum(1 for r in pos if r["size"] == 9)
rng = random.Random(0); w9 = w9raw[:]; rng.shuffle(w9)
need9 = max(0, n7 - n9p)                       # top up 9x9 toward the 7x7 count (capped by availability)
train_rows = list(pos) + w11 + w9[:need9]      # 7: positives; 11: +all filtered wtl; 9: +filtered wtl
rng.shuffle(train_rows)
# clean held-out dev (NOT in train): the 7x7 works_too_long (unfiltered) + any leftover 9x9 supplements
unused = byw[7] + w9[need9:]; rng.shuffle(unused)
dev_rows = unused[:64]
print("TRAIN by size:", dict(sorted(collections.Counter(r["size"] for r in train_rows).items())), "| total", len(train_rows))
print("DEV   by size:", dict(sorted(collections.Counter(r["size"] for r in dev_rows).items())), "| total", len(dev_rows))

ds = DatasetDict({
    "train": Dataset.from_list(train_rows),
    "dev":   Dataset.from_list(dev_rows),
})

def render(row):
    # add_generation_prompt=False -> include the assistant turn as the target
    return {"text": tokenizer.apply_chat_template(row["messages"], tokenize=False,
                                                   add_generation_prompt=False)}

ds = ds.map(render, remove_columns=[c for c in ds["train"].column_names if c != "text"])
print(ds)
print("\n--- one rendered example (head) ---\n", ds["train"][0]["text"][:600])

# --- token-length preflight: confirm nothing gets truncated at max_seq_length ---
_lens = [len(tokenizer(t, add_special_tokens=False)["input_ids"]) for t in ds["train"]["text"]]
_mx = max(_lens); _over = sum(1 for n in _lens if n > max_seq_length)
print(f"\ntoken lengths: max={_mx}  mean={sum(_lens)//len(_lens)}  rows>{max_seq_length}: {_over}")
if _over:
    print(f"WARNING: {_over} rows exceed max_seq_length={max_seq_length} and WILL be truncated")
    print(f"         (loss then trains on cut-off programs). Raise max_seq_length (cell 3) above {_mx}.")
else:
    print(f"OK: all {len(_lens)} rows fit (max {_mx} <= {max_seq_length}); no truncation.")

## 6. Response-only loss
Qwen renders the assistant turn after `<|im_start|>assistant\n`. Masking up to that marker means gradients flow only through the generated program, not the (fixed) system contract or user prompt.

In [ ]:
from trl import DataCollatorForCompletionOnlyLM
response_template = "<|im_start|>assistant\n"   # Qwen chat-template assistant marker
collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)
# sanity: confirm the marker tokenizes and is found in a sample
assert response_template in ds["train"][0]["text"], "assistant marker not found — check template"

## 7. Train (dev = in-training validation; eval stays untouched)

In [ ]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_eval_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=learning_rate,
    lr_scheduler_type=lr_scheduler_type,
    warmup_ratio=warmup_ratio,
    weight_decay=weight_decay,
    logging_steps=logging_steps,
    optim="paged_adamw_8bit",   # QLoRA-standard 8-bit optimizer: less optimizer memory -> room for a bigger batch
    group_by_length=True,       # bucket similar-length rows so short rows are not padded up to the batch max (matters once batch>1)
    use_liger_kernel=False,     # True (+ pip install liger-kernel) fuses lm_head+CE: no full-logits OOM, allows a bigger batch
    bf16=bf16_ok,
    fp16=not bf16_ok,
    gradient_checkpointing=gradient_checkpointing,   # auto-set in cell 3
    gradient_checkpointing_kwargs={"use_reentrant": True},   # must match cell 4 (fixes Qwen3 CheckpointError)
    max_length=max_seq_length,   # canonical arg; max_seq_length is deprecated/ignored
    dataset_text_field="text",
    packing=False,   # required for response-only masking
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="tensorboard",
)

# guard: confirm the trainer will actually cap at max_seq_length (trl renamed
# max_seq_length -> max_length; a wrong arg would silently truncate to a small default).
_ml, _msl = getattr(args, "max_length", None), getattr(args, "max_seq_length", None)
print(f"SFTConfig max length -> max_length={_ml}, max_seq_length={_msl}")
assert max_seq_length in (_ml, _msl), (
    f"neither SFTConfig max length == {max_seq_length}; arg-name mismatch would silently truncate")

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["dev"],
    processing_class=tokenizer,   # tokenize the `text` field with our configured tokenizer
    peft_config=peft_config,
    data_collator=collator,
)
trainer.train()

## 8. Save LoRA adapter + merged fp16 model

In [ ]:
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print("saved LoRA adapter to", adapter_dir)

# merge to a standalone fp16 model for inference / GGUF export
from peft import PeftModel
import torch, gc
del model, trainer; gc.collect(); torch.cuda.empty_cache()
base = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16,
                                            device_map={"": 0}, trust_remote_code=True)
merged = PeftModel.from_pretrained(base, adapter_dir).merge_and_unload()
merged.save_pretrained(adapter_dir + "-merged")
tokenizer.save_pretrained(adapter_dir + "-merged")
print("saved merged model to", adapter_dir + "-merged")

## 9. Persist to Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/slm_ckpt
# {adapter_dir} is interpolated by IPython from the Python namespace at runtime
!cp -r {adapter_dir} {adapter_dir}-merged /content/drive/MyDrive/slm_ckpt/ 2>/dev/null; echo saved

## Next
Your trained artifacts are in Drive (`MyDrive/slm_ckpt/`): the LoRA adapter (`qwen3-4b-crossword-qlora`) and the merged fp16 model (`…-merged`) for inference / GGUF export.

Eval is run **separately** in `colab_eval_non_hardcoded.ipynb` (harness-scored, vs Claude EVAL 3). The goal is the base-vs-tuned comparison in `GAP_ANALYSIS.md`: score the tuned model on the pristine held-out `eval.jsonl` through the sandbox+scorer and compare against unaugmented Opus (~5–7% valid) — target is high pass@1.